# exp71: distilled fine-tune of the seed314159 secondary on all 199 videos (TRAINING ONLY, never submit)

This notebook only trains. It writes no submission file, and it must never be submitted to the competition.

**Recipe:** team y3uanm's public notebook `y3uanm/notebook0afc5663f6` (v1), kept verbatim where possible.
- Teacher and init: the pilkwang seed314159 secondary (sha256 `9bac2fa0...`).
- Trainable: `unet.decoder_blocks[-1]`, `unet.head` and `detect_head`. BatchNorm stays frozen because `train()` is forced to eval mode.
- Detection loss: softplus at the GT centres, plus 0.5 x BCE to the frozen teacher on non-GT voxels, plus 0.1 x feature MSE to the teacher.
- Settings: AdamW lr 1e-5, batch 1, fp32, seed 314159, no validation, no augmentations, all 199 annotated train videos.

**What exp71 adds (the recipe cells are unchanged):**
- A pin on the trainer sha (`c4f63177...`, the file x138 pins).
- A 40-iteration timing probe. After it, the real run uses `max_iters = min(19701, int(4.0*3600 / s_per_it))`, capped at one pass over the loader (18,707 windows), so it is never more than v1's single epoch.
- `num_workers=4`. The sample order is unchanged.
- A non-caching replacement for `itertools.cycle`, so host RAM does not fill up.
- A tqdm refresh interval of 60 s.
- A hard stop at 6 h of training wall time, which saves the student.
- A 300 s DataLoader timeout. If the probe's loader fails, it is retried with `num_workers=0`. If the real run's loader fails, the student is saved and the stop is flagged in the log and the manifest.
- A sanity cell.
- An export cell that writes the x138 secondary layout and a manifest with `model.weight_sha256`.

**Output:** `/kaggle/working/secft/{ARTIFACT_MANIFEST.json, weights/unet_transformer/split_0/{edge_predictor_best.pth, config.json}}` and `/kaggle/working/exp71_train_log.json`.


In [ ]:
import copy
import hashlib
import json
import shutil
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import copy
import torch
import torch.nn.functional as functional

import time

input_root = Path("/kaggle/input")

# 新模型包上传到 Kaggle 后，修改这一处即可。
# 第一次使用原始模型时改成：
# SECONDARY_SLUG = "biohub-temporal-unet3d-seed314159-v1"
SECONDARY_SLUG = "biohub-temporal-unet3d-seed314159-v1"


def choose_existing_dir(name, candidates):
    for path in candidates:
        if path.is_dir():
            print(f"{name}: {path}", flush=True)
            return path
    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"{name} not found. Checked:\n{checked}"
    )


path_t0 = time.perf_counter()

support = choose_existing_dir(
    "support",
    [
        input_root / "datasets" / "pilkwang"
        / "biohub-tracking-support-pack-50ep-v1",
        input_root / "datasets" / "hftams"
        / "biohub-tracking-support-pack-50ep-v1",
        input_root / "biohub-tracking-support-pack-50ep-v1",
    ],
)

secondary = choose_existing_dir(
    "secondary",
    [
        input_root / "datasets" / "pilkwang" / SECONDARY_SLUG,
        input_root / "datasets" / "hftams" / SECONDARY_SLUG,
        input_root / SECONDARY_SLUG,
    ],
)

competition = choose_existing_dir(
    "competition",
    [
        input_root / "competitions"
        / "biohub-cell-tracking-during-development",
        input_root / "biohub-cell-tracking-during-development",
    ],
)

assert (competition / "train").is_dir(), (
    f"train directory not found: {competition / 'train'}"
)

print(
    f"Path initialization: {time.perf_counter() - path_t0:.2f}s",
    flush=True,
)

run_dir = Path(tempfile.mkdtemp(prefix="secondary_ft_", dir="/kaggle/working"))
repo_dir = run_dir / "repo"
initial_weights_root = run_dir / "initial_weights"

print("support:", support)
print("secondary:", secondary)
print("competition:", competition)
print("run_dir:", run_dir)

# 只安装训练脚本实际需要的顶层包。
# --no-deps 是关键：避免 tracksdata 自动拉取 imagecodecs==2026.6.26。
runtime_packages = [
    "tracksdata",
    "polars==1.42.0",
    "polars-runtime-32==1.42.0",
    "zarr==3.2.1",
    "geff==1.2.0.1.1",
    "geff-spec==1.1.1",
    "numcodecs==0.15.1",
    "donfig==0.8.1.post1",
    "bidict==0.23.1",
    "ilpy==0.6.0",
    "pyscipopt==6.2.1",
    "dask",
    "blosc2",
    "pyarrow",
    "rustworkx",
    "sqlalchemy",
]

wheels_dir = support / "wheels"
assert wheels_dir.is_dir(), f"wheels directory not found: {wheels_dir}"

install_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-index",
    "--no-deps",
    "--force-reinstall",
    "--find-links",
    str(wheels_dir),
    *runtime_packages,
]

print("Installing runtime packages with --no-deps...")
print(" ".join(install_command))
subprocess.run(install_command, check=True)

# 复制推理仓库源码。
repo_source = support / "repo"
repo_archive = support / "repo.zip"

if repo_source.is_dir():
    shutil.copytree(repo_source, repo_dir)
elif repo_archive.is_file():
    shutil.unpack_archive(str(repo_archive), str(run_dir))
    if not repo_dir.exists():
        extracted = next(
            path for path in run_dir.iterdir()
            if path.is_dir() and path.name not in {"initial_weights"}
        )
        extracted.rename(repo_dir)
else:
    raise FileNotFoundError("Support pack contains neither repo/ nor repo.zip")

# 复制副模型权重。
secondary_weights_source = secondary / "weights"
secondary_weights_archive = secondary / "weights.zip"

if secondary_weights_source.is_dir():
    shutil.copytree(secondary_weights_source, initial_weights_root)
elif secondary_weights_archive.is_file():
    shutil.unpack_archive(str(secondary_weights_archive), str(initial_weights_root))
else:
    raise FileNotFoundError("Secondary artifact contains neither weights/ nor weights.zip")

secondary_weights = (
    initial_weights_root
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
secondary_config = secondary_weights.with_name("config.json")

assert secondary_weights.is_file(), secondary_weights
assert secondary_config.is_file(), secondary_config

sys.path[:0] = [
    str(repo_dir / "scripts"),
    str(repo_dir / "src"),
]

train_script = repo_dir / "scripts" / "train_unet_transformer.py"
# [exp71] guard (not in y3uanm v1): patch only the exact support-pack trainer that x138 pins.
assert hashlib.sha256(train_script.read_bytes()).hexdigest() == (
    "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea"
), "support-pack train_unet_transformer.py is not the pinned version"
train_source = train_script.read_text(encoding="utf-8")


def replace_once(source, old, new, label):
    count = source.count(old)
    assert count == 1, (
        f"{label}: expected exactly one match, found {count}"
    )
    return source.replace(old, new, 1)


# 1. 给 train() 增加 validate 参数。
train_source = replace_once(
    train_source,
    """    pool_kernel_um: float = 5.0,
    data_parallel: bool = True,
) -> UNetNodeTransformer:""",
    """    pool_kernel_um: float = 5.0,
    data_parallel: bool = True,
    validate: bool = True,
) -> UNetNodeTransformer:""",
    "train signature",
)

# 3. validate=False 时不加载验证数据。
train_source = replace_once(
    train_source,
    """    train_video_data = _load(train_files, "train")
    test_video_data = _load(test_files, "test")
""",
    """    train_video_data = _load(train_files, "train")
    test_video_data = _load(test_files, "test") if validate else []
""",
    "validation data loading",
)


# 4. 不创建验证 Dataset。
train_source = replace_once(
    train_source,
    "    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes)\n",
    "    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes) if validate else None\n",
    "validation dataset",
)


# 5. 不创建验证 DataLoader。
train_source = replace_once(
    train_source,
    """    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0, pin_memory=False,
        generator=g, worker_init_fn=worker_init_fn,
    )
""",
    """    test_loader = (
        DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            prefetch_factor=2 if num_workers > 0 else None,
            persistent_workers=num_workers > 0,
            pin_memory=False,
            generator=g,
            worker_init_fn=worker_init_fn,
        )
        if validate
        else None
    )
""",
    "validation dataloader",
)


# 6. validate=False 时跳过完整 evaluate()。
train_source = replace_once(
    train_source,
    """        t0 = time.monotonic()
        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)
        test_time = time.monotonic() - t0

        score = test_acc * test_recall
        is_best = score >= best_score

        if is_best:
            best_score = score
            # Normalise any DataParallel "unet.module." prefix to "unet." so the
            # checkpoint loads on a single GPU (e.g. in the prediction script).
            torch.save(
                {k.replace("unet.module.", "unet.", 1): v for k, v in model.state_dict().items()},
                save_path,
            )
""",
    """        if validate:
            t0 = time.monotonic()
            test_loss, test_acc, test_recall = evaluate(
                model,
                test_loader,
                device,
                pool_kernel_um=pool_kernel_um,
            )
            test_time = time.monotonic() - t0

            score = test_acc * test_recall
            is_best = score >= best_score
        else:
            test_loss = float("nan")
            test_acc = float("nan")
            test_recall = float("nan")
            test_time = 0.0
            score = 0.0
            is_best = True

        if is_best:
            best_score = score
            # Normalize any DataParallel prefix before saving.
            torch.save(
                {
                    k.replace("unet.module.", "unet.", 1): v
                    for k, v in model.state_dict().items()
                },
                save_path,
            )
""",
    "evaluation block",
)


# 7. 训练结束时输出正确的模式说明。
train_source = replace_once(
    train_source,
    """    print(f"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}", flush=True)
""",
    """    if validate:
        print(
            f"\\nBest score (acc*recall): {best_score:.4f}, "
            f"saved to {save_path}",
            flush=True,
        )
    else:
        print(
            f"\\nTrain-only mode: latest model saved to {save_path}",
            flush=True,
        )
""",
    "training summary",
)


compile(train_source, str(train_script), "exec")
train_script.write_text(train_source, encoding="utf-8")

import inspect
import train_unet_transformer as trainer

assert Path(trainer.__file__).resolve() == train_script.resolve(), (
    f"Loaded wrong trainer: {trainer.__file__}"
)
assert "validate" in inspect.signature(trainer.train).parameters, (
    "Loaded trainer has no validate parameter. Restart the kernel."
)
assert torch.cuda.is_available(), "CUDA GPU is required"

print("Training script patched and imported.", flush=True)
print("trainer:", trainer.__file__, flush=True)
print("train signature:", inspect.signature(trainer.train), flush=True)
print("torch:", torch.__version__, flush=True)
print("CUDA:", torch.cuda.get_device_name(0), flush=True)
print("secondary weights:", secondary_weights, flush=True)
print("secondary config:", secondary_config, flush=True)


In [ ]:
import numpy as np
import zarr

train_root = competition / "train"
sample = next(
    path for path in sorted(train_root.glob("*.zarr"))
    if path.with_suffix(".geff").exists()
)
dataset = trainer.open_dataset(
    sample, require_tracks=True, load_image=False, downsample=(1, 4, 4)
)
image = zarr.open_group(str(dataset.zarr_path), mode="r")["0"]
frame = np.asarray(image[0, :, ::4, ::4])
assert frame.ndim == 3 and frame.size > 0 and np.isfinite(frame).all()
print("读取成功:", sample.name, "图像:", frame.shape)
print("标注节点数:", len(dataset.tracks.node_attrs()))


In [ ]:
initial_weights = secondary_weights
actual_initial_sha256 = hashlib.sha256(initial_weights.read_bytes()).hexdigest()
expected_initial_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
assert actual_initial_sha256 == expected_initial_sha256, "Initial weight SHA256 mismatch"
config = json.loads(initial_weights.with_name("config.json").read_text(encoding="utf-8"))
initial_state = torch.load(initial_weights, map_location="cpu", weights_only=True)
data_dir = competition / "train"
names = sorted(
    path.stem for path in data_dir.glob("*.zarr")
    if path.with_suffix(".geff").exists()
)
assert names, f"No annotated training datasets found in {data_dir}"
split = {"split": 0, "train": names, "test": []}
splits_path = run_dir / "splits.json"
splits_path.write_text(json.dumps([split], indent=2), encoding="utf-8")
print(f"Train: {len(names)}; local validation disabled", flush=True)


In [ ]:
BaseModel = trainer.UNetNodeTransformer
reference = {}
class SecondaryFineTune(BaseModel):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.load_state_dict(initial_state, strict=True)
        reference["teacher"] = copy.deepcopy(self).cuda().eval().requires_grad_(False)
        self.requires_grad_(False)
        for module in (self.unet.decoder_blocks[-1], self.unet.head, self.detect_head): module.requires_grad_(True)
    def train(self, mode=True):
        return super().train(False)
    def encode(self, imgs):
        features, logits = super().encode(imgs)
        if torch.is_grad_enabled():
            with torch.no_grad(): teacher_features, teacher_logits = reference["teacher"].encode(imgs)
            reference["logits"] = iter(teacher_logits)
            reference["feature_loss"] = functional.mse_loss(features.float(), teacher_features.float()) / teacher_features.float().square().mean().clamp_min(1e-6)
        return features, logits
def sparse_detection_loss(logits, coords, mask, neg_weight=0.01):
    """结合标注中心监督、未标注区域蒸馏与共享特征约束。"""
    teacher_prob = next(reference["logits"]).float().sigmoid()
    points = coords[mask].long().clamp_min(0).minimum(torch.tensor(logits.shape[2:], device=logits.device) - 1)
    positive = torch.zeros_like(logits, dtype=torch.bool)
    positive[mask.nonzero()[:, 0], 0, points[:, 0], points[:, 1], points[:, 2]] = True
    positive_loss = functional.softplus(-logits.float())[positive].sum() / positive.sum().clamp_min(1)
    distillation = functional.binary_cross_entropy_with_logits(logits.float(), teacher_prob, reduction="none")
    return positive_loss + 0.5 * distillation[~positive].sum() / (~positive).sum().clamp_min(1) + 0.1 * reference["feature_loss"]


In [ ]:
# [exp71] Run control + instrumentation. NOT part of y3uanm v1: the recipe cells above are v1 verbatim
# (plus the trainer-sha guard in the setup cell). Nothing here changes the loss, the optimiser or the data.
import datetime
import functools
import gc
import math
import os
import statistics
import traceback

# Defaults = the Kaggle run. The EXP71_* environment overrides exist only for the local CPU smoke test.
PROBE_ITERS = int(os.environ.get("EXP71_PROBE_ITERS", "40"))
PROBE_WARMUP_SKIP = int(os.environ.get("EXP71_PROBE_WARMUP_SKIP", "10"))
TRAIN_BUDGET_S = float(os.environ.get("EXP71_TRAIN_BUDGET_H", "4.0")) * 3600.0
HARD_STOP_S = float(os.environ.get("EXP71_HARD_STOP_H", "6.0")) * 3600.0
NUM_WORKERS = int(os.environ.get("EXP71_NUM_WORKERS", "4"))
DATALOADER_TIMEOUT_S = float(os.environ.get("EXP71_DATALOADER_TIMEOUT_S", "300"))
PROGRESS_EVERY = int(os.environ.get("EXP71_PROGRESS_EVERY", "500"))
EXPECTED_TRAIN_VIDEOS = int(os.environ.get("EXP71_EXPECTED_TRAIN_VIDEOS", "199"))
# Spec cap: 199 videos x 99 windows (T=100, W=2). The trainer skips windows with an empty GT frame, so the real
# loader is shorter (18,707 counted locally: 44b6 6,315 + 6bba 12,392); the probe cell also caps at one pass.
EPOCH_ITERS = 19701

assert len(names) == EXPECTED_TRAIN_VIDEOS, (len(names), EXPECTED_TRAIN_VIDEOS)
assert split["train"] == names and split["test"] == [], "exp71 trains on ALL annotated videos"

EXP71_LOG = {
    "kernel": "vitaliineibauer/biohub-exp71-secft-all",
    "recipe": "y3uanm notebook0afc5663f6 (v1): teacher-distilled fine-tune of the pilkwang seed314159 secondary",
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "torch": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
    "n_visible_gpus": torch.cuda.device_count(),
    "train_videos": len(names),
    "parent_sha256": expected_initial_sha256,
    "knobs": {
        "probe_iters": PROBE_ITERS,
        "probe_warmup_skip": PROBE_WARMUP_SKIP,
        "train_budget_s": TRAIN_BUDGET_S,
        "hard_stop_s": HARD_STOP_S,
        "num_workers": NUM_WORKERS,
        "dataloader_timeout_s": DATALOADER_TIMEOUT_S,
        "epoch_iters": EPOCH_ITERS,
        "lr": 1e-5,
        "batch_size": 1,
        "seed": 314159,
    },
    "runs": {},
}
_current_run = {}


class _WallClockStop(RuntimeError):
    """Raised from the batch iterator when the hard wall-clock stop of a run is reached."""


def _nocache_cycle(loader):
    """Drop-in for trainer._cycle.

    trainer._cycle is itertools.cycle, which keeps a reference to EVERY batch it yields (~0.55 MB per
    window -> ~10 GB of host RAM over 19,701 windows when max_iters is set). This version re-iterates
    the loader instead (same order on the first pass: same seeded sampler), and stamps each yielded
    batch for the timing probe, the progress lines and the hard wall-clock stop.
    """
    run = _current_run
    run["loader_len"] = len(loader)
    stamps = run["stamps"]
    while True:
        for batch in loader:
            now = time.monotonic()
            stop_at = run.get("hard_stop_at")
            if stop_at is not None and now > stop_at:
                EXP71_LOG["runs"][run["label"]].update(hard_stopped=True, stop_reason="hard_wall_clock")
                raise _WallClockStop(
                    f"{run['label']}: hard wall-clock stop after {len(stamps)} iterations"
                )
            stamps.append(now)
            k = len(stamps)
            if PROGRESS_EVERY > 0 and k % PROGRESS_EVERY == 0:
                elapsed = now - stamps[0]
                spi = elapsed / max(k - 1, 1)
                target = run.get("max_iters") or k
                print(
                    f"[exp71] {run['label']}: it {k}/{target} | {elapsed / 60:.1f} min | "
                    f"{spi:.3f} s/it | eta {(target - k) * spi / 60:.1f} min",
                    flush=True,
                )
            yield batch


if not hasattr(trainer, "_exp71_orig_train_epoch"):
    trainer._exp71_orig_train_epoch = trainer.train_epoch
    trainer._exp71_orig_tqdm = trainer.tqdm
    trainer._exp71_orig_DataLoader = trainer.DataLoader


def _dataloader_with_timeout(*args, **kwargs):
    """trainer.DataLoader with a per-batch timeout when workers are used (the trainer passes timeout=0 = wait
    forever): a stuck worker then raises instead of silently burning the session up to the 12 h limit."""
    if kwargs.get("num_workers", 0) > 0 and DATALOADER_TIMEOUT_S > 0:
        kwargs.setdefault("timeout", DATALOADER_TIMEOUT_S)
    return trainer._exp71_orig_DataLoader(*args, **kwargs)


def _is_loader_failure(exc):
    """DataLoader timeouts / dead workers surface as RuntimeError at next(batch_iter), before any compute."""
    return isinstance(exc, RuntimeError) and "DataLoader" in str(exc)


def _timed_train_epoch(model, *args, **kwargs):
    """Wraps trainer.train_epoch: keeps a handle on the student (for the hard stop) and logs the epoch."""
    _current_run["model"] = model
    t0 = time.monotonic()
    out = trainer._exp71_orig_train_epoch(model, *args, **kwargs)
    stats = EXP71_LOG["runs"][_current_run["label"]]
    stats["train_epoch_wall_s"] = round(time.monotonic() - t0, 1)
    stats["avg_edge_loss"] = float(out[0])
    stats["avg_det_loss"] = float(out[1])
    return out


trainer._cycle = _nocache_cycle
trainer.train_epoch = _timed_train_epoch
trainer.DataLoader = _dataloader_with_timeout
# The trainer's tqdm bars refresh every 0.1 s on stderr (~140k updates in a 4 h run); 60 s keeps the log small.
trainer.tqdm = functools.partial(trainer._exp71_orig_tqdm, mininterval=60.0)


def _begin_run(label, max_iters, hard_stop_s=None):
    _current_run.clear()
    _current_run.update(
        label=label,
        max_iters=max_iters,
        stamps=[],
        hard_stop_at=None if hard_stop_s is None else time.monotonic() + hard_stop_s,
    )
    EXP71_LOG["runs"][label] = {"max_iters": max_iters, "hard_stopped": False, "stop_reason": None}


def _end_run(wall_s):
    """Summarise the per-iteration stamps of the current run; returns the per-iteration durations."""
    stamps = _current_run["stamps"]
    diffs = [b - a for a, b in zip(stamps, stamps[1:])]
    stats = EXP71_LOG["runs"][_current_run["label"]]
    stats["n_iters"] = len(stamps)
    stats["wall_s_incl_data_loading"] = round(wall_s, 1)
    stats["loader_len"] = _current_run.get("loader_len")
    if diffs:
        stats["seconds_per_iter_mean"] = round(statistics.fmean(diffs), 4)
        stats["seconds_per_iter_by_1000"] = [
            round(statistics.fmean(diffs[i:i + 1000]), 4) for i in range(0, len(diffs), 1000)
        ]
    return diffs


print(
    f"exp71 control: probe {PROBE_ITERS} it (skip {PROBE_WARMUP_SKIP}), budget {TRAIN_BUDGET_S / 3600:.2f} h, "
    f"hard stop {HARD_STOP_S / 3600:.2f} h, num_workers {NUM_WORKERS} (loader timeout {DATALOADER_TIMEOUT_S:.0f} s), "
    f"{len(names)} train videos",
    flush=True,
)


In [ ]:
# [exp71] Timing probe: the v1 training call with max_iters=PROBE_ITERS (same model, loss, data order),
# written to a throw-away weights root. Its steady-state seconds/iteration sets the real run length:
#     MAX_ITERS = min(19701, int(4.0 * 3600 / seconds_per_iter)), and never more than one pass over the loader
#     (18,707 windows on the full train set), i.e. at most v1's single epoch.
probe_weights_root = run_dir / "probe_weights"


def _run_probe(num_workers):
    _begin_run("probe", PROBE_ITERS)
    EXP71_LOG["runs"]["probe"]["num_workers"] = num_workers
    trainer.UNetNodeTransformer = SecondaryFineTune
    trainer.compute_detection_loss = sparse_detection_loss
    trainer.WEIGHTS_PATH = probe_weights_root
    t0 = time.monotonic()
    probe = trainer.train(
        data_dir=data_dir,
        fold=0,
        splits_file=splits_path,
        method="unet_transformer",
        n_epochs=1,
        lr=1e-5,
        batch_size=1,
        num_workers=num_workers,
        seed=314159,
        unet_out_channels=config["unet_out_channels"],
        unet_layers=config["unet_layers"],
        downsample=tuple(config["downsample"]),
        window_size=config["window_size"],
        det_loss_weight=1.0,
        max_iters=PROBE_ITERS,
        validate=False,
        augmentations=None,
        data_parallel=False,
        pool_kernel_um=float(config.get("pool_kernel_um", 5.0)),
    )
    return probe, _end_run(time.monotonic() - t0)


_probe_failure = None
try:
    probe_model, probe_diffs = _run_probe(NUM_WORKERS)
except RuntimeError as exc:
    if NUM_WORKERS == 0 or not _is_loader_failure(exc):
        raise
    traceback.print_exc()
    _probe_failure = repr(exc)
if _probe_failure is not None:  # retried outside the except block so the failed loader/workers are released
    EXP71_LOG["probe_worker_failure"] = _probe_failure
    print(f"WARNING: DataLoader failure with num_workers={NUM_WORKERS}; retrying the probe with num_workers=0", flush=True)
    NUM_WORKERS = 0
    reference.clear()
    _current_run.clear()
    gc.collect()
    torch.cuda.empty_cache()
    probe_model, probe_diffs = _run_probe(NUM_WORKERS)
EXP71_LOG["knobs"]["num_workers"] = NUM_WORKERS
assert len(probe_diffs) >= 2, f"timing probe produced too few iterations: {probe_diffs}"

probe_skip = min(PROBE_WARMUP_SKIP, len(probe_diffs) // 2)
probe_steady = probe_diffs[probe_skip:]
seconds_per_iter = statistics.fmean(probe_steady)
n_windows = int(_current_run["loader_len"])
budget_iters = int(TRAIN_BUDGET_S / seconds_per_iter)
MAX_ITERS = min(EPOCH_ITERS, budget_iters)
if n_windows != EPOCH_ITERS:
    # Expected: 18,707 on the full train set (windows with an empty GT frame are skipped). Never exceed one
    # pass over the loader (= v1's single epoch).
    print(f"NOTE: the loader has {n_windows} windows (cap {EPOCH_ITERS}); capping MAX_ITERS at one pass", flush=True)
    MAX_ITERS = min(MAX_ITERS, n_windows)
MAX_ITERS = max(1, MAX_ITERS)

EXP71_LOG["runs"]["probe"].update(
    warmup_skipped=probe_skip,
    seconds_per_iter_steady=round(seconds_per_iter, 4),
    seconds_per_iter_steady_median=round(statistics.median(probe_steady), 4),
    seconds_per_iter_steady_max=round(max(probe_steady), 4),
    warmup_diffs_s=[round(d, 3) for d in probe_diffs[:probe_skip]],
)
EXP71_LOG["max_iters_rule"] = {
    "formula": "min(19701, int(TRAIN_BUDGET_S / seconds_per_iter)), capped at the loader length",
    "train_budget_s": TRAIN_BUDGET_S,
    "seconds_per_iter": round(seconds_per_iter, 4),
    "budget_iters": budget_iters,
    "epoch_iters": EPOCH_ITERS,
    "loader_len": n_windows,
    "max_iters": MAX_ITERS,
    "epoch_fraction": round(MAX_ITERS / n_windows, 4),
    "projected_train_s": round(MAX_ITERS * seconds_per_iter, 1),
}
print(
    f"timing probe: {len(probe_diffs) + 1} it; steady {seconds_per_iter:.4f} s/it "
    f"(median {statistics.median(probe_steady):.4f}, max {max(probe_steady):.4f}, first {probe_skip} skipped); "
    f"probe wall incl. data loading {EXP71_LOG['runs']['probe']['wall_s_incl_data_loading'] / 60:.1f} min",
    flush=True,
)
print(
    f"MAX_ITERS = {MAX_ITERS}  (= min({EPOCH_ITERS}, int({TRAIN_BUDGET_S:.0f} / {seconds_per_iter:.4f}) = "
    f"{budget_iters}); {MAX_ITERS / n_windows:.1%} of one epoch; projected training "
    f"{MAX_ITERS * seconds_per_iter / 3600:.2f} h",
    flush=True,
)

# Free the probe completely: student, teacher copy, last autograd graph, loader workers, weights on disk.
del probe_model
reference.clear()
_current_run.clear()
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(probe_weights_root, ignore_errors=True)
print(f"after probe cleanup: CUDA memory allocated {torch.cuda.memory_allocated() / 2**20:.0f} MiB", flush=True)


In [ ]:
# y3uanm v1 cell 6 (the real run). [exp71] changes: num_workers=NUM_WORKERS (v1: 0; same sample order,
# no augmentations), max_iters=MAX_ITERS from the timing probe (v1: None = one full epoch), and a hard
# wall-clock stop / DataLoader-failure path that saves the current student exactly like train() would
# (both flagged in the log and the manifest). Everything else is verbatim.
_begin_run("train", MAX_ITERS, hard_stop_s=HARD_STOP_S)
_t0 = time.monotonic()
trainer.UNetNodeTransformer = SecondaryFineTune
trainer.compute_detection_loss = sparse_detection_loss
trainer.WEIGHTS_PATH = run_dir / "weights"
try:
    model = trainer.train(
        data_dir=data_dir,
        fold=0,
        splits_file=splits_path,
        method="unet_transformer",
        n_epochs=1,
        lr=1e-5,
        batch_size=1,
        num_workers=NUM_WORKERS,
        seed=314159,
        unet_out_channels=config["unet_out_channels"],
        unet_layers=config["unet_layers"],
        downsample=tuple(config["downsample"]),
        window_size=config["window_size"],
        det_loss_weight=1.0,
        max_iters=MAX_ITERS,
        validate=False,
        augmentations=None,
        data_parallel=False,
        pool_kernel_um=float(config.get("pool_kernel_um", 5.0)),
    )
except (_WallClockStop, RuntimeError) as stop:
    if not isinstance(stop, _WallClockStop):
        if not _is_loader_failure(stop) or "model" not in _current_run:
            raise
        traceback.print_exc()
        EXP71_LOG["runs"]["train"].update(hard_stopped=True, stop_reason=f"dataloader_failure: {stop!r}")
    print(f"WARNING: {stop!r}; saving the current student state the way train() does", flush=True)
    model = _current_run["model"]
    torch.save(
        {k.replace("unet.module.", "unet.", 1): v for k, v in model.state_dict().items()},
        trainer.WEIGHTS_PATH / "unet_transformer/split_0/edge_predictor_best.pth",
    )
_end_run(time.monotonic() - _t0)
_current_run.pop("model", None)
print(f"[exp71] real run: {EXP71_LOG['runs']['train']}", flush=True)
candidate = trainer.WEIGHTS_PATH / "unet_transformer/split_0/edge_predictor_best.pth"
model.load_state_dict(torch.load(candidate, map_location="cpu", weights_only=True), strict=True)
candidate.with_name("config.json").write_text(json.dumps(config, indent=2))
print("Candidate:", candidate, "\nSHA256:", hashlib.sha256(candidate.read_bytes()).hexdigest())


In [ ]:
# [exp71] Sanity (hard asserts): only parameters of the v1-trainable modules moved vs the teacher, every
# tensor is finite, the new sha differs from the parent, and the file strict-loads the way
# predict_unet_transformer.load_model loads BIOHUB_SECONDARY_WEIGHTS in x138.
new_state = torch.load(candidate, map_location="cpu", weights_only=True)
assert list(new_state) == list(initial_state), "state-dict keys differ from the teacher"
_last_block = len(model.unet.decoder_blocks) - 1
TRAINABLE_PREFIXES = (f"unet.decoder_blocks.{_last_block}.", "unet.head.", "detect_head.")
changed, unchanged_trainable, param_deltas = [], [], {}
for key, old in initial_state.items():
    new = new_state[key]
    assert new.shape == old.shape and new.dtype == old.dtype, f"shape/dtype changed: {key}"
    if new.is_floating_point():
        assert torch.isfinite(new).all(), f"non-finite values in {key}"
    if torch.equal(new, old):
        if key.startswith(TRAINABLE_PREFIXES):
            unchanged_trainable.append(key)
        continue
    changed.append(key)
    delta = new.double() - old.double()
    param_deltas[key] = {
        "max_abs": float(delta.abs().max()),
        "rel_l2": float(delta.norm() / old.double().norm().clamp_min(1e-12)),
    }
outside = [key for key in changed if not key.startswith(TRAINABLE_PREFIXES)]
assert not outside, f"parameters outside {TRAINABLE_PREFIXES} changed: {outside}"
assert changed, "no parameter changed: the fine-tune did not update the model"
unexpected_frozen = [
    key for key in unchanged_trainable
    if not key.endswith(("running_mean", "running_var", "num_batches_tracked"))
]
if unexpected_frozen:
    print("WARNING: trainable weights that did not move:", unexpected_frozen, flush=True)

candidate_sha256 = hashlib.sha256(candidate.read_bytes()).hexdigest()
assert candidate_sha256 != expected_initial_sha256, "new weights are byte-identical to the parent"

_candidate_config = json.loads(candidate.with_name("config.json").read_text(encoding="utf-8"))
assert _candidate_config == config, (_candidate_config, config)
_strict_check = BaseModel(
    unet=trainer.TemporalUNet3D(
        in_channels=1,
        out_channels=_candidate_config["unet_out_channels"],
        layers=_candidate_config["unet_layers"],
    ),
    unet_out_channels=_candidate_config["unet_out_channels"],
    pos_feat_dim=4 * trainer._POS_EMBED_DIM,
)
_strict_check.load_state_dict(new_state)  # strict=True, same construction as load_model()
del _strict_check

EXP71_LOG["sanity"] = {
    "candidate_sha256": candidate_sha256,
    "parent_sha256": expected_initial_sha256,
    "trainable_prefixes": list(TRAINABLE_PREFIXES),
    "changed_keys": changed,
    "unchanged_trainable_keys": unchanged_trainable,
    "param_deltas": param_deltas,
    "all_finite": True,
}
print(f"sanity OK: {len(changed)} tensors changed, all under {TRAINABLE_PREFIXES}; all finite; strict load OK")
for key in changed:
    print(f"  {key:45s} max|d|={param_deltas[key]['max_abs']:.3e}  rel_l2={param_deltas[key]['rel_l2']:.3e}")
print("new SHA256:", candidate_sha256, "\nparent   :", expected_initial_sha256, flush=True)


# [exp71] Diagnostic only (never blocks the export): teacher vs student peak counts on 8 fixed TRAIN windows
# (in-sample), with the trainer's own peak finder (detect_and_match) at p >= 0.5 and p >= 0.965.
def _exp71_detection_proxy(n_per_prefix=4):
    device = next(model.parameters()).device
    teacher = reference["teacher"]
    thresholds = {"p0.5": 0.0, "p0.965": math.log(0.965 / 0.035)}
    picks = []
    for prefix in ("44b6", "6bba"):
        group = [n for n in names if n.startswith(prefix)]
        picks += group[:: max(1, len(group) // n_per_prefix)][:n_per_prefix]
    totals = {"gt": 0}
    rows = []
    for name in picks:
        video_meta, windows = trainer.load_dataset_windows(
            data_dir / name,
            window_size=config["window_size"],
            downsample=tuple(config["downsample"]),
        )
        item = trainer.FrameWindowDataset([(video_meta, windows)])[len(windows) // 2]
        imgs = item["imgs"].to(device, dtype=torch.float32)[None]
        coords = item["coords"].to(device)[None]
        masks = item["masks"].to(device)[None]
        image_shape = tuple(int(v) for v in item["image_shape"].tolist())
        voxel_size = tuple(float(v) for v in item["voxel_size"].tolist())
        with torch.no_grad():
            _, student_logits = model.encode(imgs)
            _, teacher_logits = teacher.encode(imgs)
        row = {"video": name, "t_start": int(item["t_start"]), "gt": int(masks.sum())}
        totals["gt"] += row["gt"]
        for who, logits in (("teacher", teacher_logits), ("student", student_logits)):
            for tname, tlogit in thresholds.items():
                n_det = n_hit = 0
                for i in range(imgs.shape[1]):
                    _, _, det_mask, matches = trainer.detect_and_match(
                        logits[i], coords[:, i], masks[:, i], image_shape,
                        det_threshold=tlogit,
                        pool_kernel_um=float(config.get("pool_kernel_um", 5.0)),
                        voxel_size=voxel_size,
                        frame_index=i,
                        window_size=imgs.shape[1],
                    )
                    n_det += int(det_mask.sum())
                    n_hit += int((matches[0] >= 0).sum())
                row[f"{who}_{tname}_det"] = n_det
                row[f"{who}_{tname}_gt_hit"] = n_hit
                totals[f"{who}_{tname}_det"] = totals.get(f"{who}_{tname}_det", 0) + n_det
                totals[f"{who}_{tname}_gt_hit"] = totals.get(f"{who}_{tname}_gt_hit", 0) + n_hit
        rows.append(row)
    for tname in thresholds:
        t_det, s_det = totals[f"teacher_{tname}_det"], totals[f"student_{tname}_det"]
        totals[f"student_over_teacher_{tname}_det"] = round(s_det / max(t_det, 1), 4)
        print(
            f"  {tname}: peaks teacher {t_det} -> student {s_det} (x{s_det / max(t_det, 1):.3f}); "
            f"GT hit teacher {totals[f'teacher_{tname}_gt_hit']}/{totals['gt']} -> "
            f"student {totals[f'student_{tname}_gt_hit']}/{totals['gt']}",
            flush=True,
        )
    return {"note": "in-sample train windows; trainer peak finder, not the x138 pipeline", "totals": totals, "rows": rows}


try:
    print("detection proxy (teacher vs student, 8 train windows):", flush=True)
    EXP71_LOG["detection_proxy"] = _exp71_detection_proxy()
except Exception as exc:  # diagnostic only
    traceback.print_exc()
    EXP71_LOG["detection_proxy"] = {"error": repr(exc)}


In [ ]:
# [exp71] LAST CELL. Writes the dataset payload in the layout of pilkwang/biohub-temporal-unet3d-seed314159-v1
# that x138 code cell 3 reads (_find_secondary_artifact_root -> manifest["model"]["weight_sha256"];
# copy_or_extract_tree(<manifest dir>/weights) -> unet_transformer/split_0/{edge_predictor_best.pth, config.json};
# sha256 check of the copied weights), then removes every scratch dir from /kaggle/working.
#   /kaggle/working/secft/ARTIFACT_MANIFEST.json
#   /kaggle/working/secft/weights/unet_transformer/split_0/edge_predictor_best.pth
#   /kaggle/working/secft/weights/unet_transformer/split_0/config.json
#   /kaggle/working/exp71_train_log.json          (small log, outside the dataset folder)
working_root = Path("/kaggle/working")
secft_dir = working_root / "secft"
if secft_dir.exists():
    shutil.rmtree(secft_dir)
secft_weights = secft_dir / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
secft_weights.parent.mkdir(parents=True)
shutil.copyfile(candidate, secft_weights)
# Architecture is unchanged: ship the teacher's own config.json bytes (same 5 keys load_model reads).
shutil.copyfile(secondary_config, secft_weights.with_name("config.json"))
assert json.loads(secft_weights.with_name("config.json").read_text(encoding="utf-8")) == config

secft_sha256 = hashlib.sha256(secft_weights.read_bytes()).hexdigest()
assert secft_sha256 == candidate_sha256, (secft_sha256, candidate_sha256)
assert secft_sha256 != expected_initial_sha256

teacher_manifest = json.loads((secondary / "ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
assert teacher_manifest["model"]["weight_sha256"] == expected_initial_sha256
compatibility = copy.deepcopy(teacher_manifest.get("compatibility", {}))
compatibility.pop("gate_diagnostics", None)  # refers to full_frame_center files this artifact does not ship
compatibility.get("models", {}).pop("full_frame_center", None)

train_stats = EXP71_LOG["runs"]["train"]
n_iters = int(train_stats["n_iters"])
model_block = {
    "config": config,
    "method": "unet_transformer",
    "weight_path": "weights/unet_transformer/split_0/edge_predictor_best.pth",
    "weight_bytes": secft_weights.stat().st_size,
    "weight_sha256": secft_sha256,
    "parent_sha256": expected_initial_sha256,
    "training": {
        "method": "secondary_teacher_distill_ft_y3uanm_v1",
        "init": "pilkwang/biohub-temporal-unet3d-seed314159-v1 weights/unet_transformer/split_0/edge_predictor_best.pth",
        "trainable": ["unet.decoder_blocks[-1]", "unet.head", "detect_head"],
        "batchnorm": "frozen (train() forced to eval mode)",
        "det_loss": "softplus(-logit) at GT centres + 0.5*BCE(student, teacher sigmoid) on non-GT voxels "
                    "+ 0.1*normalised feature MSE to the teacher",
        "edge_loss": "unchanged trainer edge loss through the frozen transformer",
        "det_loss_weight": 1.0,
        "optimizer": "AdamW (torch defaults besides lr)",
        "learning_rate": 1e-5,
        "batch_size": 1,
        "precision": "fp32",
        "seed": 314159,
        "augmentations": [],
        "validation": None,
        "train_datasets": len(names),
        "loader_len": train_stats.get("loader_len"),
        "max_iters": MAX_ITERS,
        "n_iters": n_iters,
        "epoch_fraction": round(n_iters / max(int(train_stats.get("loader_len") or 0), 1), 4),
        "hard_stopped": bool(train_stats.get("hard_stopped")),
        "stop_reason": train_stats.get("stop_reason"),
        "num_workers": NUM_WORKERS,
        "seconds_per_iter_probe": EXP71_LOG["max_iters_rule"]["seconds_per_iter"],
        "seconds_per_iter_mean": train_stats.get("seconds_per_iter_mean"),
        "train_wall_s": train_stats.get("wall_s_incl_data_loading"),
    },
}
manifest = {
    "artifact_name": "biohub-secft-all-v1",
    "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source": "private training kernel vitaliineibauer/biohub-exp71-secft-all (y3uanm notebook0afc5663f6 recipe)",
    "parent_artifact": "pilkwang/biohub-temporal-unet3d-seed314159-v1",
    "compatibility": compatibility,
    "contents": {
        "weights": {"files": 2, "bytes": sum(p.stat().st_size for p in secft_weights.parent.iterdir())},
        "repo_included": False,
        "wheels_included": False,
    },
    "model": model_block,
    "models": {"unet_transformer": model_block},
    "n_iters": n_iters,
    "notes": [
        "Drop-in secondary for x138-style notebooks: same architecture/config as the parent, only the weights differ.",
        "x138 hard-codes the parent sha (code cell 3 _secondary_expected_sha256); set it to model.weight_sha256 and "
        "point BIOHUB_SECONDARY_ARTIFACT_MANIFEST at this file.",
    ],
}
(secft_dir / "ARTIFACT_MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

# Emulate x138 code cell 3 on the written folder.
_info = json.loads((secft_dir / "ARTIFACT_MANIFEST.json").read_text())
assert str(_info.get("model", {}).get("weight_sha256", "")) == secft_sha256
with tempfile.TemporaryDirectory() as _tmp:
    _copied = Path(_tmp) / "secondary_seed_weights"
    shutil.copytree(secft_dir / "weights", _copied)
    _w = _copied / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
    assert _w.is_file() and _w.with_name("config.json").is_file()
    assert hashlib.sha256(_w.read_bytes()).hexdigest() == secft_sha256
print("x138 loader emulation OK", flush=True)

EXP71_LOG["artifact"] = {
    "dir": str(secft_dir),
    "weight_sha256": secft_sha256,
    "parent_sha256": expected_initial_sha256,
    "n_iters": n_iters,
    "max_iters": MAX_ITERS,
}
EXP71_LOG["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
(working_root / "exp71_train_log.json").write_text(json.dumps(EXP71_LOG, indent=2) + "\n", encoding="utf-8")

# Cleanup: repo copy, teacher copy, trained-weights scratch, splits (all under run_dir).
shutil.rmtree(run_dir)
_total = 0
print("final /kaggle/working:")
for _p in sorted(working_root.rglob("*")):
    if _p.is_file():
        _total += _p.stat().st_size
        print(f"  {_p.relative_to(working_root)}  {_p.stat().st_size:,} B")
print(f"  total {_total / 2**20:.1f} MiB")
if _total > 64 * 2**20:
    print("WARNING: /kaggle/working is larger than expected", flush=True)
print("NEW SECONDARY SHA256 (for the x138 sha anchor / --patch secswap):", secft_sha256)
print(
    "n_iters:", n_iters, "| max_iters:", MAX_ITERS, "| loader_len:", train_stats.get("loader_len"),
    "| hard_stopped:", train_stats.get("hard_stopped"), "| stop_reason:", train_stats.get("stop_reason"),
)
